# MNIST AGOP Analysis

Comprehensive analysis of 12 completed MNIST experiments comparing:
- **Optimizers**: AdamW, Muon, SGD
- **Weight Decays**: 0.01, 0.1, 0.5, 1.0

This notebook analyzes:
1. Performance metrics (accuracy, grokking outcomes)
2. AGOP metric evolution and relationship to grokking
3. Statistical comparisons between optimizers
4. Normalized overlay plots of AGOP metrics with train/test accuracy


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

sys.path.append(str(Path.cwd()))
from analysis_utils import (
    load_all_experiments, generate_summary_table, classify_grokking,
    compute_time_to_grok, statistical_comparison, filter_experiments,
    plot_agop_comparison, create_comparison_heatmap, smooth_series,
    detect_phase_transitions, compute_correlation
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Paths
RESULTS_DIR = Path('../results/mnist')
FIGURES_DIR = Path('./figures/mnist')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {RESULTS_DIR.absolute()}")
print(f"Figures will be saved to: {FIGURES_DIR.absolute()}")


## Load All MNIST Experiments


In [ ]:
experiments = load_all_experiments(RESULTS_DIR)
print(f"Loaded {len(experiments)} experiments")

summary_df = generate_summary_table(experiments)
print(f"\nSummary of experiments:")
display(summary_df.sort_values(['optimizer', 'weight_decay']))


## Quick Statistics


In [ ]:
# Overall statistics
total = len(summary_df)
grokked = summary_df['grokked'].sum()
grok_rate = grokked / total * 100

print(f"="*80)
print(f"MNIST DATASET - OVERALL STATISTICS")
print(f"="*80)
print(f"Total experiments: {total}")
print(f"Grokked: {grokked} ({grok_rate:.1f}%)")
print(f"Failed to grok: {total - grokked} ({100-grok_rate:.1f}%)")
print()

# By optimizer
print(f"By Optimizer:")
print(f"-"*80)
for opt in summary_df['optimizer'].unique():
    opt_df = summary_df[summary_df['optimizer'] == opt]
    opt_grok = opt_df['grokked'].sum()
    opt_total = len(opt_df)
    print(f"  {opt.upper()}: {opt_grok}/{opt_total} grokked ({opt_grok/opt_total*100:.1f}%)")

print(f"="*80)

# Store grok rate for later use
grok_rate_value = grok_rate


---
# Section 0: Training Dynamics - Accuracy and Loss Curves

**First Analytical Insight**: Visualize training and test accuracy/loss over epochs for all completed experiments. 
Red dashed vertical lines indicate the "grokking epoch" where test performance catches up with train performance.


In [ ]:
def detect_grokking_epoch(train_acc, test_acc, grok_threshold=0.85, min_epoch=100):
    """
    Detect the epoch where grokking occurs (test accuracy exceeds threshold).
    
    Args:
        train_acc: Array of training accuracies (unused, kept for compatibility)
        test_acc: Array of test accuracies
        grok_threshold: Test accuracy threshold to define grokking (default 0.85)
        min_epoch: Minimum epoch to start checking (avoid early noise)
    
    Returns:
        Grokking epoch (int) or None if no grokking detected
    """
    if len(test_acc) < min_epoch:
        return None
    
    # Find the first epoch where test accuracy exceeds the grokking threshold
    for i in range(min_epoch, len(test_acc)):
        if test_acc[i] > grok_threshold:
            return i
    
    return None

print("Grokking detection function loaded.")
print(f"Grokking definition: test_acc > 0.85")


## Generate Training Dynamics Plots for All Experiments


In [ ]:
# Create a subdirectory for training dynamics plots
TRAIN_DYNAMICS_DIR = FIGURES_DIR / 'training_dynamics'
TRAIN_DYNAMICS_DIR.mkdir(parents=True, exist_ok=True)

grokking_summary = []

for exp_name, exp_data in experiments.items():
    history = exp_data.get('history', {})
    config = exp_data.get('config', {})
    
    # Check if we have the required data
    required_keys = ['train_acc', 'test_acc', 'train_loss', 'test_loss']
    if not all(key in history for key in required_keys):
        print(f"Skipping {exp_name}: missing required metrics")
        continue
    
    # Extract data
    epochs = np.array(history.get('epoch', range(len(history['test_acc']))))
    train_acc = np.array(history['train_acc'])
    test_acc = np.array(history['test_acc'])
    train_loss = np.array(history['train_loss'])
    test_loss = np.array(history['test_loss'])
    
    # Detect grokking epoch
    grok_epoch_idx = detect_grokking_epoch(train_acc, test_acc)
    grok_epoch = epochs[grok_epoch_idx] if grok_epoch_idx is not None else None
    
    # Store summary
    grokking_summary.append({
        'experiment': exp_name,
        'optimizer': config.get('optimizer', 'unknown'),
        'weight_decay': config.get('weight_decay', 0),
        'grok_epoch': grok_epoch if grok_epoch else -1,
        'final_test_acc': test_acc[-1],
        'final_train_acc': train_acc[-1]
    })
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Plot 1: Accuracy
    ax1.plot(epochs, train_acc, 'b-', linewidth=2, label='Train Accuracy', alpha=0.8)
    ax1.plot(epochs, test_acc, 'orange', linewidth=2, label='Test Accuracy', alpha=0.8)
    
    if grok_epoch is not None:
        ax1.axvline(x=grok_epoch, color='red', linestyle='--', linewidth=2, 
                   label=f'Grokking Epoch: {grok_epoch}', alpha=0.7)
        ax1.text(grok_epoch, 0.5, f'  Epoch {grok_epoch}', 
                rotation=90, verticalalignment='center', fontsize=10, color='red')
    
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title('Train vs Test Accuracy', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='best')
    ax1.grid(alpha=0.3)
    ax1.set_ylim([-0.05, 1.05])
    
    # Plot 2: Loss
    ax2.plot(epochs, train_loss, 'b-', linewidth=2, label='Train Loss', alpha=0.8)
    ax2.plot(epochs, test_loss, 'orange', linewidth=2, label='Test Loss', alpha=0.8)
    
    if grok_epoch is not None:
        ax2.axvline(x=grok_epoch, color='red', linestyle='--', linewidth=2, 
                   label=f'Grokking Epoch: {grok_epoch}', alpha=0.7)
    
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title('Train vs Test Loss', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10, loc='best')
    ax2.grid(alpha=0.3)
    ax2.set_yscale('log')
    
    # Overall title
    opt = config.get('optimizer', 'unknown').upper()
    wd = config.get('weight_decay', 0)
    status = "GROKKED" if grok_epoch else "NO GROK"
    title_color = 'green' if grok_epoch else 'darkred'
    
    fig.suptitle(f'MNIST: {opt} | WD={wd} | {status}', 
                fontsize=16, fontweight='bold', color=title_color)
    plt.tight_layout()
    
    # Save figure
    save_path = TRAIN_DYNAMICS_DIR / f'{exp_name}_dynamics.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Generated plot: {save_path.name}")

print(f"\nGenerated {len(grokking_summary)} training dynamics plots.")

# Create summary DataFrame
grokking_df = pd.DataFrame(grokking_summary)
print("\nGrokking Summary:")
display(grokking_df.sort_values(['optimizer', 'weight_decay']))


## Training Dynamics Overview (Grid of All Experiments)


In [ ]:
# Create overview plot with all experiments
n_experiments = len(experiments)
n_cols = 3
n_rows = int(np.ceil(n_experiments / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

exp_list = list(experiments.items())
exp_list.sort(key=lambda x: (x[1]['config'].get('optimizer', ''), x[1]['config'].get('weight_decay', 0)))

for idx, (exp_name, exp_data) in enumerate(exp_list):
    ax = axes[idx]
    history = exp_data.get('history', {})
    config = exp_data.get('config', {})
    
    if 'test_acc' not in history or 'train_acc' not in history:
        ax.axis('off')
        continue
    
    epochs = np.array(history.get('epoch', range(len(history['test_acc']))))
    train_acc = np.array(history['train_acc'])
    test_acc = np.array(history['test_acc'])
    
    # Detect grokking
    grok_epoch_idx = detect_grokking_epoch(train_acc, test_acc)
    grok_epoch = epochs[grok_epoch_idx] if grok_epoch_idx is not None else None
    
    # Plot
    ax.plot(epochs, train_acc, 'b-', linewidth=1.5, label='Train', alpha=0.7)
    ax.plot(epochs, test_acc, 'orange', linewidth=1.5, label='Test', alpha=0.7)
    
    if grok_epoch is not None:
        ax.axvline(x=grok_epoch, color='red', linestyle='--', linewidth=2, alpha=0.7)
        ax.text(grok_epoch, 0.05, f'Epoch {grok_epoch}', 
               rotation=90, verticalalignment='bottom', fontsize=8, color='red',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    # Title
    opt = config.get('optimizer', 'unknown')
    wd = config.get('weight_decay', 0)
    status = "GROKKED ✓" if grok_epoch else "NO GROK"
    title_color = 'green' if grok_epoch else 'darkred'
    ax.set_title(f'{opt.upper()} | WD={wd}\n{status}', 
                fontsize=10, fontweight='bold', color=title_color)
    
    ax.set_xlabel('Epoch', fontsize=9)
    ax.set_ylabel('Accuracy', fontsize=9)
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)
    ax.set_ylim([-0.05, 1.05])

# Hide unused subplots
for idx in range(len(exp_list), len(axes)):
    axes[idx].axis('off')

fig.suptitle('MNIST Training Dynamics Overview: All Experiments', 
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_dynamics_overview.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved overview figure: training_dynamics_overview.png")


## Focused Analysis: Grokking by Optimizer

Deep dive into configurations that achieve grokking.
We'll examine how different optimizers and weight decay settings affect the grokking phenomenon.


In [ ]:
# Group experiments by optimizer
mnist_by_opt = {}
for opt in ['adamw', 'muon', 'sgd']:
    opt_exps = {name: data for name, data in experiments.items() 
                if data['config'].get('optimizer') == opt}
    if len(opt_exps) > 0:
        mnist_by_opt[opt] = sorted(opt_exps.items(), 
                                   key=lambda x: x[1]['config'].get('weight_decay', 0))

# Detailed breakdown by optimizer
for opt in ['adamw', 'muon', 'sgd']:
    if opt not in mnist_by_opt:
        continue
    
    sorted_exps = mnist_by_opt[opt]
    
    print(f"\n{opt.upper()} MNIST experiments: {len(sorted_exps)}")
    
    # Count grokking
    grok_count = sum(1 for _, data in sorted_exps 
                     if classify_grokking(data))
    print(f"  Grokked: {grok_count}/{len(sorted_exps)}")
    
    for exp_name, exp_data in sorted_exps:
        wd = exp_data['config'].get('weight_decay', 0)
        history = exp_data['history']
        
        if 'test_acc' not in history:
            continue
        
        test_acc = np.array(history['test_acc'])
        train_acc = np.array(history['train_acc'])
        
        grok_epoch_idx = detect_grokking_epoch(train_acc, test_acc)
        epochs = np.array(history.get('epoch', range(len(test_acc))))
        grok_epoch = epochs[grok_epoch_idx] if grok_epoch_idx is not None else None
        
        final_test = test_acc[-1]
        final_train = train_acc[-1]
        
        if grok_epoch:
            print(f"  WD={wd:5.2f}: GROKKED at epoch {grok_epoch:5d} | Final train/test: {final_train:.4f}/{final_test:.4f}")
        else:
            print(f"  WD={wd:5.2f}: NO GROK                  | Final train/test: {final_train:.4f}/{final_test:.4f}")


---
# Section 1: AGOP Metrics Analysis

Now let's examine how AGOP metrics evolve differently across optimizers.
We'll compare configurations that grok vs those that don't with **normalized overlay plots**.


In [ ]:
# Define AGOP metrics to analyze
agop_metrics = [
    ('agop_variation_collapse_ratio', 'VCR (λ₁ / Σλᵢ)', 'Variation Collapse Ratio'),
    ('agop_eigengap', 'Eigengap (λ₁ - λ₂)', 'Eigengap'),
    ('agop_trace', 'Trace (Σλᵢ)', 'Trace'),
    ('agop_spectral_radius', 'Spectral Radius (λ₁)', 'Spectral Radius'),
]

def normalize_series(series):
    """Normalize a series to [0, 1] range using min-max normalization."""
    s_min = np.min(series)
    s_max = np.max(series)
    if s_max - s_min < 1e-10:  # Avoid division by zero
        return np.zeros_like(series)
    return (series - s_min) / (s_max - s_min)

print(f"Analyzing AGOP metrics for MNIST experiments by optimizer")
print(f"{'='*80}\n")


## AGOP Metrics with Train/Test Accuracy Overlays (Normalized)

For each optimizer and each AGOP metric, we create vertically stacked plots showing:
- **Blue solid line**: AGOP metric (normalized)
- **Green dashed line**: Train accuracy (normalized)
- **Orange dotted line**: Test accuracy (normalized)
- **Red vertical line**: Grokking epoch (if applicable)

This allows us to see temporal relationships between AGOP metrics and learning dynamics.


In [ ]:
# For each optimizer, create overlay plots for each AGOP metric
for opt, sorted_exps in mnist_by_opt.items():
    print(f"\nProcessing {opt.upper()} MNIST experiments...")
    
    for metric_name, metric_label, metric_title in agop_metrics:
        # Collect all data for this optimizer
        all_data = []
        
        for exp_name, exp_data in sorted_exps:
            history = exp_data.get('history', {})
            config = exp_data.get('config', {})
            
            # Check if all required data exists
            if metric_name not in history or 'train_acc' not in history or 'test_acc' not in history:
                continue
            
            epochs = np.array(history.get('epoch', range(len(history.get('test_acc', [])))))
            metric_values = np.array(history[metric_name])
            train_acc = np.array(history['train_acc'])
            test_acc = np.array(history['test_acc'])
            
            # Align lengths
            min_len = min(len(epochs), len(metric_values), len(train_acc), len(test_acc))
            epochs = epochs[:min_len]
            metric_values = metric_values[:min_len]
            train_acc = train_acc[:min_len]
            test_acc = test_acc[:min_len]
            
            # Detect grokking
            grok_epoch_idx = detect_grokking_epoch(train_acc, test_acc)
            grok_epoch = epochs[grok_epoch_idx] if grok_epoch_idx is not None else None
            
            # Smooth and normalize
            metric_smooth = smooth_series(metric_values, window=10)
            train_smooth = smooth_series(train_acc, window=10)
            test_smooth = smooth_series(test_acc, window=10)
            
            metric_norm = normalize_series(metric_smooth)
            train_norm = normalize_series(train_smooth)
            test_norm = normalize_series(test_smooth)
            
            all_data.append({
                'exp_name': exp_name,
                'config': config,
                'epochs': epochs,
                'metric_norm': metric_norm,
                'train_norm': train_norm,
                'test_norm': test_norm,
                'grok_epoch': grok_epoch
            })
        
        if not all_data:
            print(f"  No data available for {metric_title}, skipping...")
            continue
        
        max_epochs = max([d['epochs'][-1] for d in all_data if len(d['epochs']) > 0])
        
        # Create vertically stacked subplots (one per weight decay)
        n_subplots = len(all_data)
        fig, axes = plt.subplots(n_subplots, 1, figsize=(14, 5 * n_subplots))
        
        # Handle single subplot case
        if n_subplots == 1:
            axes = [axes]
        
        for idx, data_dict in enumerate(all_data):
            ax = axes[idx]
            
            epochs = data_dict['epochs']
            metric_norm = data_dict['metric_norm']
            train_norm = data_dict['train_norm']
            test_norm = data_dict['test_norm']
            grok_epoch = data_dict['grok_epoch']
            config = data_dict['config']
            wd = config.get('weight_decay', 0)
            
            # Plot all three normalized signals
            ax.plot(epochs, metric_norm, linewidth=2.5, color='blue', alpha=0.8, 
                   label=f'{metric_label} (norm)', linestyle='-')
            ax.plot(epochs, train_norm, linewidth=2.0, color='green', alpha=0.7, 
                   label='Train Acc (norm)', linestyle='--')
            ax.plot(epochs, test_norm, linewidth=2.0, color='orange', alpha=0.7, 
                   label='Test Acc (norm)', linestyle=':')
            
            # Mark grokking epoch if it exists
            if grok_epoch is not None:
                ax.axvline(x=grok_epoch, color='red', linestyle='--', linewidth=2.5, alpha=0.8, 
                          label=f'Grokking (Epoch {grok_epoch})')
                status = "GROKKED ✓"
                title_color = 'green'
            else:
                status = "NO GROK"
                title_color = 'darkred'
            
            # Formatting with unified axes
            ax.set_ylabel('Normalized Value', fontsize=12, fontweight='bold')
            ax.set_title(f'Weight Decay = {wd}  —  {status}', 
                        fontsize=13, fontweight='bold', color=title_color, pad=10, loc='left')
            ax.legend(fontsize=9, loc='best', ncol=2)
            ax.grid(alpha=0.3, linestyle='--')
            
            # Set unified axis limits
            ax.set_xlim([0, max_epochs])
            ax.set_ylim([-0.05, 1.05])
            
            # Only show x-label on bottom plot
            if idx == len(all_data) - 1:
                ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
            else:
                ax.set_xlabel('')
        
        # Overall title
        fig.suptitle(f'{metric_title} with Train/Test Accuracy (All Normalized)\nMNIST + {opt.upper()} Across Weight Decays', 
                    fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout()
        
        # Save figure
        safe_metric_name = metric_name.replace('_', '-')
        plt.savefig(FIGURES_DIR / f'mnist_{opt}_{safe_metric_name}_with_accuracy_overlay.png', 
                   dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"  Generated overlay panel for {metric_title}")


---
# Section 2: AGOP Metric Comparisons Across Optimizers

Compare how each AGOP metric behaves across different optimizers to identify patterns.


In [ ]:
# Create comparison plots for each metric across all optimizers
for metric_name, metric_label, metric_title in agop_metrics:
    print(f"\nGenerating cross-optimizer comparison for {metric_title}...")
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    for opt_idx, (opt, sorted_exps) in enumerate(mnist_by_opt.items()):
        ax = axes[opt_idx]
        
        for exp_name, exp_data in sorted_exps:
            history = exp_data.get('history', {})
            config = exp_data.get('config', {})
            
            if metric_name not in history:
                continue
            
            epochs = np.array(history.get('epoch', range(len(history.get('test_acc', [])))))
            metric_values = np.array(history[metric_name])
            wd = config.get('weight_decay', 0)
            
            # Smooth the metric
            metric_smooth = smooth_series(metric_values, window=10)
            
            # Plot
            ax.plot(epochs[:len(metric_smooth)], metric_smooth, linewidth=2, 
                   label=f'WD={wd}', alpha=0.7)
        
        ax.set_title(f'{opt.upper()}', fontsize=13, fontweight='bold')
        ax.set_ylabel(metric_label, fontsize=11)
        ax.legend(fontsize=9, loc='best')
        ax.grid(alpha=0.3)
        
        if opt_idx == 2:
            ax.set_xlabel('Epoch', fontsize=11)
    
    fig.suptitle(f'{metric_title} - Comparison Across Optimizers (MNIST)', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    safe_metric_name = metric_name.replace('_', '-')
    plt.savefig(FIGURES_DIR / f'mnist_{safe_metric_name}_optimizer_comparison.png', 
               dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"  Saved: mnist_{safe_metric_name}_optimizer_comparison.png")


---
# Section 3: Correlation Analysis

Compute correlations between AGOP metrics and test accuracy to understand which metrics
are most predictive of generalization performance.


In [ ]:
# Compute correlations for each experiment
correlation_results = []

for exp_name, exp_data in experiments.items():
    history = exp_data.get('history', {})
    config = exp_data.get('config', {})
    
    if 'test_acc' not in history:
        continue
    
    test_acc = np.array(history['test_acc'])
    
    for metric_name, _, metric_title in agop_metrics:
        if metric_name not in history:
            continue
        
        metric_values = np.array(history[metric_name])
        
        # Align lengths
        min_len = min(len(test_acc), len(metric_values))
        
        # Compute correlation
        corr = np.corrcoef(test_acc[:min_len], metric_values[:min_len])[0, 1]
        
        correlation_results.append({
            'experiment': exp_name,
            'optimizer': config.get('optimizer', 'unknown'),
            'weight_decay': config.get('weight_decay', 0),
            'metric': metric_title,
            'correlation': corr
        })

corr_df = pd.DataFrame(correlation_results)

# Create pivot table for heatmap
pivot_data = {}
for metric_title in corr_df['metric'].unique():
    metric_data = corr_df[corr_df['metric'] == metric_title]
    pivot = metric_data.pivot_table(values='correlation', 
                                    index='optimizer', 
                                    columns='weight_decay',
                                    aggfunc='mean')
    pivot_data[metric_title] = pivot

# Plot heatmaps
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (metric_title, pivot) in enumerate(pivot_data.items()):
    ax = axes[idx]
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
                vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title(f'{metric_title} vs Test Accuracy', fontsize=12, fontweight='bold')
    ax.set_xlabel('Weight Decay', fontsize=10)
    ax.set_ylabel('Optimizer', fontsize=10)

fig.suptitle('AGOP Metric Correlations with Test Accuracy (MNIST)', 
            fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mnist_agop_correlation_heatmaps.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: mnist_agop_correlation_heatmaps.png")

# Display summary statistics
print("\nCorrelation Summary by Metric:")
print("="*80)
summary = corr_df.groupby('metric')['correlation'].agg(['mean', 'std', 'min', 'max'])
display(summary.sort_values('mean', ascending=False))


---
# Section 4: Key Findings Summary

Summarize the main findings from the MNIST analysis.


In [ ]:
print("="*80)
print("MNIST AGOP ANALYSIS - KEY FINDINGS")
print("="*80)

# Overall grokking statistics
print(f"\n1. GROKKING STATISTICS:")
print(f"   Total experiments: {len(experiments)}")
print(f"   Experiments that grokked: {grokked} ({grok_rate:.1f}%)")
print(f"   Experiments that did not grok: {total - grokked} ({100-grok_rate:.1f}%)")

# By optimizer
print(f"\n2. GROKKING BY OPTIMIZER:")
for opt in ['adamw', 'muon', 'sgd']:
    opt_df = summary_df[summary_df['optimizer'] == opt]
    if len(opt_df) > 0:
        opt_grok = opt_df['grokked'].sum()
        opt_total = len(opt_df)
        print(f"   {opt.upper()}: {opt_grok}/{opt_total} grokked ({opt_grok/opt_total*100:.1f}%)")

# Grokking timing
print(f"\n3. GROKKING TIMING (for experiments that grokked):")
grokked_exps = grokking_df[grokking_df['grok_epoch'] > 0]
if len(grokked_exps) > 0:
    print(f"   Mean grokking epoch: {grokked_exps['grok_epoch'].mean():.0f}")
    print(f"   Median grokking epoch: {grokked_exps['grok_epoch'].median():.0f}")
    print(f"   Earliest grokking: {grokked_exps['grok_epoch'].min():.0f}")
    print(f"   Latest grokking: {grokked_exps['grok_epoch'].max():.0f}")

# Final performance
print(f"\n4. FINAL PERFORMANCE:")
print(f"   Mean final test accuracy (all): {grokking_df['final_test_acc'].mean():.4f}")
print(f"   Mean final test accuracy (grokked): {grokked_exps['final_test_acc'].mean():.4f}")
non_grokked = grokking_df[grokking_df['grok_epoch'] <= 0]
if len(non_grokked) > 0:
    print(f"   Mean final test accuracy (non-grokked): {non_grokked['final_test_acc'].mean():.4f}")

# AGOP metric correlations
print(f"\n5. AGOP METRIC CORRELATIONS WITH TEST ACCURACY:")
for metric_title in corr_df['metric'].unique():
    metric_corrs = corr_df[corr_df['metric'] == metric_title]['correlation']
    print(f"   {metric_title}: mean={metric_corrs.mean():.3f}, std={metric_corrs.std():.3f}")

print(f"\n{'='*80}")
print(f"Analysis complete! All figures saved to: {FIGURES_DIR.absolute()}")
print(f"{'='*80}")
